<a href="https://colab.research.google.com/github/azka-asif/flyrank-ML/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/azka-asif/flyrank-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Signal 1 - Staleness:

My rule focuses on how long it has been since a page was last updated. The reason for this was that pages which have not been updates for a long time would be more likely to show a declining trend. I tested this using the dataset's freshness_tier buckets and printed the amount of pages in each bucket along with the decline rate.

The largest bucket which contained stale pages (91-180 days, n=9,171), did show a higher rate of decline than the freshest bucket. This supports my hypothesis. However, the oldest bucket (181+) drops back below the freshest bucket's rate, and that bucket contains only 174 pages. This factor makes me reconsider whether the reversal is reliable evidence against the rule. But I also cannot say that staleness entirley predicts decline on its own. I am giving this signal a verdict of MIXED.

Signal 2 - Visibility:

My rule also focuses on impressions_90d. This is the signal behind the quick win flag. In this case, my hypothesis was narrower than "more traffic means more greater". I wanted to check whether a page needs a minimum amount of traffic before a trend label even means  anything. I bucketed according to the dataset's impression_tier and looked at what share of each bucket has a trend_direction of "new" or "flat". This would mean that there was not enough prior traffic to calculate a real trend.

The visibilty floor removes exactly the pages where the trend label is not reliable. I am giving this signal a verdict of CONFIRMED.

Rule: A page is worth a refresh review if it has gone atleats 90 days without being updated while still receiving atleats 100 impressions in the last 90 days. Both of these conditions must be true.


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys
if "google.colab" in sys.modules:
    if not os.path.isdir("flyrank-ML"):
        !git clone https://github.com/azka-asif/flyrank-ML
    os.chdir("flyrank-ML")
else:
    while not os.path.isdir("data/raw"):
        os.chdir("..")

print(os.getcwd())

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
#signal 1 - staleness
signal1 = df.groupby("freshness_tier")["is_declining_label"].agg(["count", "mean"])
signal1.columns = ["n", "decline_rate"]
print(signal1)
#signal 2 - visibility
df["unmeasurable"] = df["trend_direction"].isin(["new", "flat"])
signal2 = df.groupby("impression_tier").agg(n=("is_declining_label", "count"),decline_rate=("is_declining_label", "mean"),pct_unmeasurable=("unmeasurable", "mean"))
print(signal2)

Cloning into 'flyrank-ML'...
remote: Enumerating objects: 144, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 144 (delta 55), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (144/144), 1.86 MiB | 2.38 MiB/s, done.
Resolving deltas: 100% (55/55), done.
/content/flyrank-ML/flyrank-ML
                    n  decline_rate
freshness_tier                     
0-30            20480      0.511377
181+              174      0.471264
31-90             175      0.588571
91-180           9171      0.611057
                     n  decline_rate  pct_unmeasurable
impression_tier                                       
excellent         1078      0.461967          0.000928
good              7205      0.586121          0.004025
low              11248      0.453947          0.288318
moderate         10469      0.614672          0.010985


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

#stale = hasn't been updated in a while
#visible = still gets a considerable amount of impressions
df["stale"] = df["days_since_last_update"] >= 90
df["visible"] = df["impressions_90d"] >= 100

#turning True/False into 1/0 so we can multiply them
df["stale"] = df["stale"].astype(int)
df["visible"] = df["visible"].astype(int)

#a page is only flagged if BOTH are true
df["flagged"] = df["stale"] * df["visible"]
# score is 0 if not flagged
df["score"] = df["flagged"] * df["impressions_90d"]

reason_codes = []
actions = []
for f in df["flagged"]:
    if f == 1:
        reason_codes.append("stale_but_visible")
        actions.append("refresh_review")
    else:
        reason_codes.append("not_flagged")
        actions.append("monitor")

df["reason_code"] = reason_codes
df["action"] = actions

df = df.sort_values("score", ascending=False)
df = df.reset_index(drop=True)
df["rank"] = df.index + 1

print("number of pages flagged:", df["flagged"].sum())
print(df[["rank", "content_id", "score", "reason_code", "action"]].head(10))

import os
os.makedirs("work/outputs", exist_ok=True)
df.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("saved csv")

number of pages flagged: 8119
   rank            content_id   score        reason_code          action
0     1  content_5fe46e04994d  517715  stale_but_visible  refresh_review
1     2  content_2dba2b1f9536  443434  stale_but_visible  refresh_review
2     3  content_2c2606c5d176  347399  stale_but_visible  refresh_review
3     4  content_cb112fce36be  309910  stale_but_visible  refresh_review
4     5  content_9532f197bbc8  309192  stale_but_visible  refresh_review
5     6  content_36ff89c8214e  295097  stale_but_visible  refresh_review
6     7  content_b28d1efd668f  286608  stale_but_visible  refresh_review
7     8  content_813e88069237  233561  stale_but_visible  refresh_review
8     9  content_c21024970297  211366  stale_but_visible  refresh_review
9    10  content_c8e9d6ab9013  208678  stale_but_visible  refresh_review
saved csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 3. Top-10 review

For each of my top 10 flagged pages: the action, why it's there, and what would make it
wrong. (My card asks for a top-10 review — top-20 is optional/bonus only.)

1. content_5fe46e04994d — action: refresh_review. Why: huge traffic (517,715
   impressions) at position 4.2, but ctr is only 0.14%, far lower than it should be at
   that position, and it's trending down. What would make it wrong: if the low ctr is
   caused by a snippet or ad sitting above it, not the content itself.

2. content_2dba2b1f9536 — action: refresh_review. Why: second-highest traffic in the
   list, well past the staleness cutoff. What would make it wrong: trend_direction is
   "stable," not "down," and word_count is already 7,676 — already long, so a refresh may
   not help much.

3. content_2c2606c5d176 — action: refresh_review. Why: same low-ctr/good-position
   pattern as row 1, and trending down. What would make it wrong: word_count is missing
   here, so I don't actually know what to change.

4. content_cb112fce36be — action: refresh_review. Why: top-5 position, transactional
   page, very low ctr, trending down. What would make it wrong: if the low ctr is
   seasonal, e.g. tied to a promo that already ended.

5. content_9532f197bbc8 — action: refresh_review. Why: position 2 but ctr is only
   0.87%, which should be much higher at that position, and it's trending down. What would
   make it wrong: if a SERP feature like People Also Ask is taking the clicks instead.

6. content_36ff89c8214e — action: refresh_review. Why: lowest ctr in the top 10
   (0.05%) at a decent position (7.3). What would make it wrong: trend is "stable," not
   "down," and ctr that low might just be a tracking issue.

7. content_b28d1efd668f — action: refresh_review. Why: lots of traffic, transactional,
   but stuck deep on page 3 (position 26.2). What would make it wrong: trend is "stable"
   and word_count is already 6,901 — position is probably not a content problem.

8. content_813e88069237 — action: refresh_review. Why: same weak position as row 7,
   but this one is actually trending down. What would make it wrong: if the whole client's
   site lost traffic in this window, not just this page.

9. content_c21024970297 — action: refresh_review. Why: good position (5.1), high
   traffic, but ctr still looks low for that position. What would make it wrong: trend is
   "stable" — this pick mostly rides on being stale and visible, not on an actual decline.

10. content_c8e9d6ab9013 — action: refresh_review. Why: ctr is exactly 0.00% with real traffic (208,678 impressions), and it's trending down — something looks broken. What would make it wrong: if 0.00% ctr is a tracking error, not a real number — should check clicks_90d directly first.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# just showing the top 10 rows I'm reviewing above, so the numbers can be checked
top10 = df.head(10)
print(top10[["rank", "content_id", "avg_position", "ctr", "trend_direction", "word_count"]].to_string())


   rank            content_id  avg_position   ctr trend_direction  word_count
0     1  content_5fe46e04994d           4.2  0.14            down         NaN
1     2  content_2dba2b1f9536          27.9  0.21          stable      7676.0
2     3  content_2c2606c5d176           4.2  0.53            down         NaN
3     4  content_cb112fce36be           5.6  0.16            down      2761.0
4     5  content_9532f197bbc8           2.0  0.87            down         NaN
5     6  content_36ff89c8214e           7.3  0.05          stable         NaN
6     7  content_b28d1efd668f          26.2  0.06          stable      6901.0
7     8  content_813e88069237          26.2  0.06            down      4610.0
8     9  content_c21024970297           5.1  0.41          stable      2874.0
9    10  content_c8e9d6ab9013           9.7  0.00            down         NaN


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: 4 of my top 10 (content_2dba2b1f9536, content_36ff89c8214e, content_b28d1efd668f, content_c21024970297) have trend_direction="stable," not "down". So according to the label these pages are not actually declining. They only rank this high because my score is raw impressions_90d and they happen to be big, stale, and visible. This shows a limit of the rule: it can't tell "big and stale" apart from "big, stale, and actually
getting worse." Two of those four are also already long articles (7,676 and 6,901 words), so "refresh for more depth" probably isn't even the correct fix for them.

Leakage check: My score only uses days_since_last_update and impressions_90d. Both of these are properties of the page itself, not an outcome. I did not use trend_direction, trend_pct, or
is_declining_label to build stale, visible, score, reason_code, or action.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

#the only two columns I actually used to build the score
score_inputs = ["days_since_last_update", "impressions_90d"]
#the label-derived columns not allowed to use as inputs
label_columns = ["trend_direction", "trend_pct", "is_declining_label"]
print("columns used to build the score:", score_inputs)
print("label-derived columns (should not be used):", label_columns)

overlap = set(score_inputs) & set(label_columns)
if len(overlap) == 0:
    print("no overlap - the score does not use any label-derived column")
else:
    print("PROBLEM - leakage found:", overlap)

columns used to build the score: ['days_since_last_update', 'impressions_90d']
label-derived columns (should not be used): ['trend_direction', 'trend_pct', 'is_declining_label']
no overlap - the score does not use any label-derived column


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.